<a href="https://colab.research.google.com/github/Fdiannz/BT4222-Project-Group-4--MIND/blob/main/step3_feature_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# step3_feature_mlp: Feature-based MLP

**Purpose:** a feature-based ranking model. For each displayed article it combines engineered user, news, context and user × news features with a learned topic embedding, and an MLP turns them into one click score. The slate is ranked by score.

**Inputs** (from `data/processed/` unless stated):
- `features_{fit,tune,eval}.parquet` from `step3_features` (one row per candidate)
- `step3_category_vocab.json` (category and subcategory IDs from the Fit articles only, 0 = unseen)
- `{fit,tune,eval}_impressions.parquet` and `news_metadata.parquet` from step 1 (click histories and article categories)
- for the training-performance check only: `step3_article_first_seen.parquet` and `data/embeddings/news_embeddings.npy` + `news_embedding_ids.parquet`

**Outputs:**
- `data/scores/step3_feature_mlp_{tune,eval,train_sample}.parquet`: one row per displayed candidate, columns `impression_id, news_id, score`. Used by step 6.
- `outputs/step3_feature_mlp/`: checkpoints, training logs, standardisation statistics and result tables.


### Inputs to the MLP (53)

| Group | Input | Size | How it is built |
|---|---|---|---|
| User | log(1 + history length) | 1 | Number of past clicks, log-scaled |
| User | User topic embedding | 24 | Average of the category (8) and subcategory (16) embeddings of the user's 50 most recent clicks |
| News | News topic embedding | 24 | Category (8) + subcategory (16) embedding of the candidate |
| News | Exposure percentile in slate | 1 | Rank of the candidate's prior display count within its full displayed slate (0–1), from `step3_features` |
| Context | log article age (hours) | 1 | Time since the article was first displayed, log-scaled |
| User × news | Content similarity | 1 | Cosine similarity between the user's average MiniLM article embedding and the candidate's |
| User × news | Category match | 1 | Share of the user's history in the candidate's category |

The user and news topic embeddings share one category table and one subcategory table, so a user's taste and an article's topic live in the same space. The five numeric inputs are standardised with Fit means and standard deviations. No user or article IDs are used, and `cold_article` is not an input (it is only used to split the results).

**Why the exposure percentile replaces the raw count:** prior display counts are counted from 9 Nov, so they are near 0 on the Fit days but much larger on the Tune and Test days. A model trained on the raw count would be tested on values it never saw. The rank within the slate keeps the same meaning ("how much has this article been shown compared with the others in this slate") on every day.

### Model
`[numeric 5 | user topic 24 | news topic 24] → Linear(53 → 64) → ReLU → Dropout 0.2 → Linear(64 → 32) → ReLU → Dropout 0.2 → Linear(32 → 1) → click score`

**Training:** each click is grouped with 4 non-clicked articles from the same impression. The 5 articles are scored, and softmax cross-entropy over the 5 scores (target = the click) is minimised with AdamW. After each epoch a fixed Tune subset is scored, the best epoch is kept, and training stops early after 2 epochs without improvement.

**Hyperparameters:** a small grid on Tune (learning rate 1e-3 / 3e-4 × hidden sizes 64→32 / 128→64), keeping the configuration with the best full Tune AUC. **Ablations** (on Tune, with the chosen configuration): without the user × news features, and without the user topic embedding.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import subprocess
from pathlib import Path

# Clone the GitHub project into Colab
PROJECT_ROOT = Path("/content/BT4222-Project-Group-4--MIND")
if not (PROJECT_ROOT / "config.py").exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git",
        str(PROJECT_ROOT)
    ], check=True)
sys.path.insert(0, str(PROJECT_ROOT))

# Shared Google Drive project folder (works with either shortcut name)
DATA_ROOT = None
for p in ["/content/drive/MyDrive/BT4222_Project", "/content/drive/MyDrive/BT4222 G4/BT4222_Project"]:
    if Path(p).is_dir():
        DATA_ROOT = Path(p)
        break
assert DATA_ROOT is not None, "Add a shortcut to BT4222_Project in My Drive (Shared with me -> Organise -> Add shortcut)"
os.environ["MIND_PROJECT_DIR"] = str(DATA_ROOT)

import config
print("Processed folder:", config.PROCESSED_DIR)

Mounted at /content/drive
Processed folder: /content/drive/MyDrive/BT4222_Project/data/processed


## 2. Settings

- **`CFG`** holds the model and training settings. `GRID` lists the hyperparameter values compared on Tune.
- **`RESUME = True`:** finished runs are loaded from their checkpoints instead of retrained.
- **`RUN_GRID`:** if `False`, only the `CFG` configuration is trained.
- **`RUN_ABLATIONS`:** set to `False` to skip section 12.

In [2]:
import os
import json
import time
import gc
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from utils.metrics import evaluate, evaluate_by_segment, label_impression_segment

PROCESSED_DIR = Path(config.PROCESSED_DIR)
DATA_DIR = Path(getattr(config, "DATA_DIR", PROCESSED_DIR.parent))
OUTPUT_DIR = Path(getattr(config, "OUTPUT_DIR", DATA_DIR.parent / "outputs"))
SCORE_DIR = Path(getattr(config, "SCORES_DIR", DATA_DIR / "scores"))
EMB_DIR = DATA_DIR / "embeddings"
RUN_DIR = OUTPUT_DIR / "step3_feature_mlp"
CACHE_DIR = Path("/content/step3_mlp_cache") if Path("/content").is_dir() else RUN_DIR / "cache"
for d in (SCORE_DIR, RUN_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
PREFIX = "step3_feature_mlp"

CFG = dict(
    max_history=50,            # most recent clicks used for the user topic embedding
    hidden=(64, 32),           # hidden layer sizes
    dropout=0.2,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=128,            # training groups (1 click + 4 non-clicks) per batch
    max_epochs=5,
    patience=2,                # stop after 2 epochs without a better Tune AUC
    n_tune_imps=20_000,        # fixed Tune subset scored after each epoch to pick the best epoch
    n_train_slate_imps=20_000, # Fit impressions re-scored on full slates (training performance)
    seed=getattr(config, "SEED", 42),
)
GRID = {"lr": [1e-3, 3e-4], "hidden": [(64, 32), (128, 64)]}
RESUME = True
RUN_GRID = True
RUN_ABLATIONS = True

# the five numeric inputs (hist_len is log-scaled when loaded)
RAW_NUMERIC_COLUMNS = ["hist_len", "cat_match", "user_article_sim", "slate_exposure_pct", "log_article_age_hours"]
NUMERICAL_COLUMNS = ["log_hist_len", "cat_match", "user_article_sim", "slate_exposure_pct", "log_article_age_hours"]
USER_NEWS_COLUMNS = ["cat_match", "user_article_sim"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])
print("Device:", device)

Device: cuda


## 3. Model

- **Embedding layers:** one category table (8-d) and one subcategory table (16-d). Index 0 (padding or unseen) is a fixed zero vector.
- **User topic embedding:** the average of the category and subcategory embeddings of the user's recent clicks, ignoring padding. Users with no history get zeros, and `log_hist_len` tells the model so.
- **News topic embedding:** the candidate's category and subcategory embeddings.
- **MLP:** the 53 inputs go through two hidden layers (ReLU, dropout) to one output, the click score.

For the ablations, `numeric` selects which numeric inputs are used and `use_user_topic=False` drops the user topic embedding. The user topic embedding is computed once per impression and shared by all its candidates (`user_topic` + `score`).

In [3]:
class FeatureMLP(nn.Module):
    def __init__(self, n_categories, n_subcategories, hidden=(64, 32), dropout=0.2,
                 numeric=NUMERICAL_COLUMNS, use_user_topic=True):
        super().__init__()
        # Shared embeddings for users' history and candidate news
        self.category_embedding = nn.Embedding(n_categories, 8, padding_idx=0)
        self.subcategory_embedding = nn.Embedding(n_subcategories, 16, padding_idx=0)
        self.numeric_idx = [NUMERICAL_COLUMNS.index(c) for c in numeric]
        self.use_user_topic = use_user_topic

        # 5 numerical + 24 user + 24 article = 53 inputs
        self.n_inputs = len(numeric) + 24 + (24 if use_user_topic else 0)
        layers, prev = [], self.n_inputs
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)
        self.architecture = "-".join(map(str, [self.n_inputs, *hidden, 1]))

    def average_history(self, history_ids, embedding):
        # Ignore padding and unknown category IDs (0)
        mask = (history_ids != 0).unsqueeze(-1)
        vectors = embedding(history_ids) * mask
        counts = mask.sum(dim=1).clamp(min=1)
        return vectors.sum(dim=1) / counts

    def user_topic(self, history_category_idx, history_subcategory_idx):
        # User topic embedding: 8 + 16 dimensions
        return torch.cat([
            self.average_history(history_category_idx, self.category_embedding),
            self.average_history(history_subcategory_idx, self.subcategory_embedding),
        ], dim=1)

    def score(self, category_idx, subcategory_idx, user_vectors, numerical_features):
        parts = [numerical_features[:, self.numeric_idx]]
        if self.use_user_topic:
            parts.append(user_vectors)
        parts += [self.category_embedding(category_idx), self.subcategory_embedding(subcategory_idx)]
        return self.mlp(torch.cat(parts, dim=1)).squeeze(-1)

    def forward(self, category_idx, subcategory_idx, history_category_idx,
                history_subcategory_idx, numerical_features):
        user = self.user_topic(history_category_idx, history_subcategory_idx)
        return self.score(category_idx, subcategory_idx, user, numerical_features)

print("Feature MLP architecture defined.")

Feature MLP architecture defined.


**Check with dummy data:** 5 candidates, each with a short click history (0 = padding).

In [4]:
torch.manual_seed(42)
model = FeatureMLP(n_categories=20, n_subcategories=300)
model.eval()
assert model.n_inputs == 53

category_idx = torch.tensor([1, 2, 3, 1, 2])
subcategory_idx = torch.tensor([10, 20, 30, 10, 20])
history_category_idx = torch.tensor([[1, 2, 3, 0], [2, 2, 0, 0], [3, 1, 2, 1], [0, 0, 0, 0], [1, 1, 2, 0]])
history_subcategory_idx = torch.tensor([[10, 20, 30, 0], [20, 20, 0, 0], [30, 10, 20, 10], [0, 0, 0, 0], [10, 10, 20, 0]])
numerical_features = torch.rand(5, 5)

with torch.no_grad():
    scores = model(category_idx, subcategory_idx, history_category_idx, history_subcategory_idx, numerical_features)
print("Inputs:", model.n_inputs, "| output shape:", tuple(scores.shape), "| scores:", scores.numpy().round(4))
assert scores.shape == (5,) and torch.isfinite(scores).all()

# ablation variants
assert FeatureMLP(20, 300, numeric=[c for c in NUMERICAL_COLUMNS if c not in USER_NEWS_COLUMNS]).n_inputs == 51
assert FeatureMLP(20, 300, use_user_topic=False).n_inputs == 29
print("MLP architecture test PASSED")

Inputs: 53 | output shape: (5,) | scores: [0.2266 0.2672 0.2051 0.2098 0.2728]
MLP architecture test PASSED


## 4. Training loss and training groups

**Loss:** each training group holds 1 clicked and 4 non-clicked articles from the same impression. Softmax over the 5 scores gives click probabilities, and cross-entropy pushes up the probability of the click.

**Groups:** every click in `features_fit` gets 4 different non-clicked articles from the same impression, drawn at random (two clicks in one impression may share non-clicks). A click with fewer than 4 non-clicks available is skipped. The click's position in the group is shuffled, and `targets` records where it is.

In [5]:
def five_way_loss(scores, clicked_positions):
    """scores: [batch_size, 5]; clicked_positions: [batch_size]"""
    assert scores.ndim == 2 and scores.shape[1] == 5
    return F.cross_entropy(scores, clicked_positions)


def five_way_groups_from_arrays(imp, news, label, rng, k=4):
    """Rows -> [n_groups, 1 + k] row indices from one impression, the click's position, and the skip count."""
    rows = np.arange(len(label))
    pos, neg = rows[label == 1], rows[label == 0]
    # one non-click per (impression, article)
    neg = neg[~pd.DataFrame({"i": imp[neg], "n": news[neg]}).duplicated().to_numpy()]
    neg = neg[np.argsort(imp[neg], kind="stable")]
    uniq, start, count = np.unique(imp[neg], return_index=True, return_counts=True)
    has_neg = np.isin(imp[pos], uniq)
    n_no_neg = int((~has_neg).sum())
    pos = pos[has_neg]
    g = np.searchsorted(uniq, imp[pos])

    # every (click, non-click) pair from the same impression
    n_pairs = count[g]
    pair_click = np.repeat(np.arange(len(pos)), n_pairs)
    offset = np.arange(n_pairs.sum()) - np.repeat(np.cumsum(n_pairs) - n_pairs, n_pairs)
    pair_neg = neg[np.repeat(start[g], n_pairs) + offset]
    ok = news[pair_neg] != news[pos[pair_click]]
    pair_click, pair_neg = pair_click[ok], pair_neg[ok]

    # random order within each click, keep the first k
    order = np.lexsort((rng.random(len(pair_click)), pair_click))
    pair_click, pair_neg = pair_click[order], pair_neg[order]
    first = np.r_[0, np.flatnonzero(np.diff(pair_click)) + 1] if len(pair_click) else np.zeros(0, int)
    rank = np.arange(len(pair_click)) - np.repeat(first, np.diff(np.r_[first, len(pair_click)]))
    pair_click, pair_neg = pair_click[rank < k], pair_neg[rank < k]
    full = np.bincount(pair_click, minlength=len(pos)) == k

    groups = np.concatenate([pos[full][:, None], pair_neg[full[pair_click]].reshape(-1, k)], axis=1)
    # shuffle so the clicked article isn't always first
    perm = np.argsort(rng.random(groups.shape), axis=1)
    groups = np.take_along_axis(groups, perm, axis=1)
    targets = np.argmax(perm == 0, axis=1)
    return groups.astype(np.int64), targets.astype(np.int64), n_no_neg + int((~full).sum())


def build_five_way_groups(df, seed=42):
    """DataFrame version (used for the check below)."""
    df = df.reset_index(drop=True)
    groups, targets, skipped = five_way_groups_from_arrays(
        pd.factorize(df["impression_id"])[0], pd.factorize(df["news_id"])[0],
        df["label"].to_numpy(), np.random.default_rng(seed))
    return df, groups, targets, skipped

**Check with dummy data:** impression 101 has 2 clicks and 5 non-clicks, 102 has 1 click and 4 non-clicks, 103 has only 2 non-clicks (skipped).

In [6]:
test_df = pd.DataFrame({
    "impression_id": [101] * 7 + [102] * 5 + [103] * 3,
    "news_id": list("ABCDEFGHIJKLMNO"),
    "label": [1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
})
test_rows, groups, targets, skipped = build_five_way_groups(test_df)
assert groups.shape == (3, 5) and targets.shape == (3,) and skipped == 1
for indices, target in zip(groups, targets):
    selected = test_rows.iloc[indices]
    assert selected["impression_id"].nunique() == 1
    assert selected["news_id"].nunique() == 5
    assert selected["label"].sum() == 1 and selected["label"].iloc[target] == 1

# one gradient step on the dummy model
model.train()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
scores = model(category_idx, subcategory_idx, history_category_idx[:1].repeat(5, 1),
               history_subcategory_idx[:1].repeat(5, 1), numerical_features).unsqueeze(0)
loss = five_way_loss(scores, torch.tensor([0]))
opt.zero_grad(); loss.backward(); opt.step()
assert torch.isfinite(loss)
print("Groups:", len(groups), "| skipped:", skipped, "| click positions:", targets.tolist(), "| loss:", round(loss.item(), 4))
print("Five-way groups and loss test PASSED")

Groups: 3 | skipped: 1 | click positions: [1, 1, 2] | loss: 1.601
Five-way groups and loss test PASSED


## 5. Helpers: Tune AUC, checkpoints and scoring

- **`mean_impression_auc`:** AUC inside each impression (ties get average ranks), averaged over impressions with both a click and a non-click. Used only to pick the best epoch; final results use the shared `utils/metrics.py`.
- **`save_best_checkpoint`:** saves the model only when the Tune AUC improves.
- **`score_rows`:** scores candidates in chunks. Each impression's user topic embedding is computed once and shared by its candidates.

In [7]:
def mean_impression_auc(impression_ids, labels, scores):
    df = pd.DataFrame({"impression_id": impression_ids, "label": labels, "score": scores})
    assert df["label"].isin([0, 1]).all() and np.isfinite(df["score"]).all()
    df["rank"] = df.groupby("impression_id", sort=False)["score"].rank(method="average")
    df["positive_rank"] = df["rank"] * df["label"]
    stats = df.groupby("impression_id", sort=False).agg(
        n=("label", "size"), n_pos=("label", "sum"), pos_rank_sum=("positive_rank", "sum"))
    stats["n_neg"] = stats["n"] - stats["n_pos"]
    valid = stats[(stats["n_pos"] > 0) & (stats["n_neg"] > 0)]
    if valid.empty:
        raise ValueError("No valid impressions for AUC.")
    aucs = (valid["pos_rank_sum"] - valid["n_pos"] * (valid["n_pos"] + 1) / 2) / (valid["n_pos"] * valid["n_neg"])
    return {"mean_auc": float(aucs.mean()), "valid_impressions": len(valid),
            "skipped_impressions": len(stats) - len(valid)}


def save_best_checkpoint(model, optimizer, epoch, tune_auc, best_auc, path):
    """Save model only when TUNE AUC improves."""
    if not np.isfinite(tune_auc):
        raise ValueError("Invalid TUNE AUC.")
    if tune_auc <= best_auc:
        return best_auc, False
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "tune_mean_auc": tune_auc, "architecture": model.architecture}, path)
    return tune_auc, True


@torch.no_grad()
def score_rows(model, data, rows=None, chunk=500_000):
    """One score per candidate row. data: cat, sub, num, imp per row; hc, hs per impression."""
    model.eval()
    rows = np.arange(len(data["label"])) if rows is None else np.asarray(rows)
    imps = np.unique(data["imp"][rows])
    user = torch.zeros((len(data["hc"]), 24), device=device)
    for s in range(0, len(imps), 100_000):
        i = torch.as_tensor(imps[s:s + 100_000], device=device, dtype=torch.long)
        hc = torch.as_tensor(data["hc"][imps[s:s + 100_000]], device=device, dtype=torch.long)
        hs = torch.as_tensor(data["hs"][imps[s:s + 100_000]], device=device, dtype=torch.long)
        user[i] = model.user_topic(hc, hs)
    out = np.empty(len(rows), dtype=np.float32)
    for s in range(0, len(rows), chunk):
        r = rows[s:s + chunk]
        out[s:s + chunk] = model.score(
            torch.as_tensor(data["cat"][r], device=device, dtype=torch.long),
            torch.as_tensor(data["sub"][r], device=device, dtype=torch.long),
            user[torch.as_tensor(data["imp"][r], device=device, dtype=torch.long)],
            torch.as_tensor(data["num"][r], device=device, dtype=torch.float32)).cpu().numpy()
    assert np.isfinite(out).all()
    return out

In [8]:
# check with small dummy data
result = mean_impression_auc(["A"] * 5 + ["B"] * 5 + ["C"] * 5,
                             [1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
                             [0.9, 0.1, 0.2, 0.3, 0.4, 0.9, 0.2, 0.1, 0.4, 0.5, 0.1, 0.2, 0.3, 0.4, 0.5])
assert np.isclose(result["mean_auc"], 0.625) and result["valid_impressions"] == 2

dummy = {"cat": np.array([1, 2, 3, 1, 2]), "sub": np.array([10, 20, 30, 10, 20]),
         "num": np.random.rand(5, 5).astype(np.float32), "imp": np.array([0, 0, 0, 1, 1]),
         "label": np.array([1, 0, 0, 0, 1]),
         "hc": history_category_idx[:2].numpy(), "hs": history_subcategory_idx[:2].numpy()}
dummy_model = FeatureMLP(20, 300).to(device)
s1 = score_rows(dummy_model, dummy, chunk=2)
with torch.no_grad():
    dummy_model.eval()
    s2 = dummy_model(*(torch.as_tensor(x, device=device) for x in (
        dummy["cat"], dummy["sub"], dummy["hc"][dummy["imp"]], dummy["hs"][dummy["imp"]])),
        torch.as_tensor(dummy["num"], device=device)).cpu().numpy()
assert np.allclose(s1, s2, atol=1e-5)

with tempfile.TemporaryDirectory() as tmp:
    _, saved = save_best_checkpoint(dummy_model, torch.optim.Adam(dummy_model.parameters()), 1,
                                    result["mean_auc"], -1.0, Path(tmp) / "test.pt")
    assert saved
print("Mean impression AUC:", result["mean_auc"], "| batched scores match per-row scores")
print("Helper tests PASSED")

Mean impression AUC: 0.625 | batched scores match per-row scores
Helper tests PASSED


## 6. Category vocabulary and Fit-only standardisation

- The category and subcategory IDs come from `step3_category_vocab.json`, built in `step3_features` from the Fit articles only, so the IDs here match the feature files. A category never seen in Fit has ID 0.
- The five numeric inputs are standardised with the **Fit** mean and standard deviation, computed in batches without loading all rows. The same values are applied to Tune and Test.

In [9]:
with open(PROCESSED_DIR / "step3_category_vocab.json", encoding="utf-8") as f:
    vocab = json.load(f)
category_to_idx = vocab["category_to_idx"]
subcategory_to_idx = vocab["subcategory_to_idx"]
n_categories = max(category_to_idx.values()) + 1
n_subcategories = max(subcategory_to_idx.values()) + 1

news_metadata = pd.read_parquet(PROCESSED_DIR / "news_metadata.parquet",
                                columns=["news_id", "category", "subcategory"]).drop_duplicates("news_id")
news_metadata["news_id"] = news_metadata["news_id"].astype(str)
news_category_lookup = dict(zip(news_metadata["news_id"], news_metadata["category"].map(category_to_idx).fillna(0).astype(int)))
news_subcategory_lookup = dict(zip(news_metadata["news_id"], news_metadata["subcategory"].map(subcategory_to_idx).fillna(0).astype(int)))
NEWS_INDEX = pd.Index(news_metadata["news_id"])

known_cat = sum(v != 0 for v in news_category_lookup.values())
known_sub = sum(v != 0 for v in news_subcategory_lookup.values())
print(f"Category vocabulary: {n_categories} | subcategory vocabulary: {n_subcategories}")
print(f"Articles with a known category: {known_cat:,} / {len(news_metadata):,} | known subcategory: {known_sub:,}")
assert known_cat > 0 and known_sub > 0

Category vocabulary: 17 | subcategory vocabulary: 219
Articles with a known category: 104,148 / 104,151 | known subcategory: 103,997


In [10]:
def get_numeric_matrix(df):
    """The five numeric inputs, with history length log-scaled."""
    x = df[RAW_NUMERIC_COLUMNS].to_numpy(dtype=np.float64, copy=True)
    x[:, 0] = np.log1p(x[:, 0])
    if not np.isfinite(x).all():
        raise ValueError("Invalid numerical feature values.")
    return x


def calculate_fit_stats(batches):
    """Mean / std over all batches (parallel update), without loading all rows."""
    count, mean, m2 = 0, np.zeros(5), np.zeros(5)
    for df in batches:
        x = get_numeric_matrix(df)
        if len(x) == 0:
            continue
        n, batch_mean = len(x), x.mean(axis=0)
        batch_m2 = ((x - batch_mean) ** 2).sum(axis=0)
        new_count = count + n
        delta = batch_mean - mean
        mean += delta * n / new_count
        m2 += batch_m2 + delta ** 2 * count * n / new_count
        count = new_count
    if count == 0:
        raise ValueError("No FIT observations found.")
    std = np.sqrt(m2 / count)
    std[std < 1e-8] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)


pf = pq.ParquetFile(PROCESSED_DIR / "features_fit.parquet")
fit_mean, fit_std = calculate_fit_stats(
    b.to_pandas() for b in pf.iter_batches(batch_size=500_000, columns=RAW_NUMERIC_COLUMNS))
pd.DataFrame({"mean": fit_mean, "std": fit_std}, index=NUMERICAL_COLUMNS).to_csv(RUN_DIR / "standardisation_fit.csv")
display(pd.DataFrame({"Fit mean": fit_mean, "Fit std": fit_std}, index=NUMERICAL_COLUMNS).round(4))

,Fit mean,Fit std
log_hist_len,3.1360,1.1488
cat_match,0.1712,0.2003
user_article_sim,0.1782,0.1209
slate_exposure_pct,0.4982,0.2921
log_article_age_hours,3.0458,0.6701


## 7. Model inputs for each split

For each split the feature file is read in batches and turned into compact arrays, saved in `/content` so a restart does not redo this:
- **per candidate row:** category ID, subcategory ID, the 5 standardised numeric inputs, label, `cold_article` (for results only), and the row's impression;
- **per impression:** the history length and the category and subcategory IDs of the 50 most recent clicks (0 = padding). The history is built **once per impression**, not once per candidate, which keeps memory and time low.

The rows keep the order of `features_{split}.parquet`, so the score files can be written by reading the same file again.

In [11]:
INPUT_COLUMNS = ["impression_id", "news_id", "label", "cold_article", "category_idx",
                 "subcategory_idx"] + RAW_NUMERIC_COLUMNS
CANDIDATE_FILES = {"fit": "fit_candidates_sampled.parquet", "tune": "tune_candidates.parquet",
                   "eval": "eval_candidates.parquet"}


def history_topics(histories, max_history=CFG["max_history"]):
    """[n_impressions, max_history] category / subcategory IDs of the most recent clicks (0 = padding)."""
    hc = np.zeros((len(histories), max_history), dtype=np.int16)
    hs = np.zeros((len(histories), max_history), dtype=np.int16)
    for i, history in enumerate(histories):
        if isinstance(history, str) and history:
            ids = history.split()[-max_history:]
            hc[i, :len(ids)] = [news_category_lookup.get(x, 0) for x in ids]
            hs[i, :len(ids)] = [news_subcategory_lookup.get(x, 0) for x in ids]
    return hc, hs


def build_split_inputs(split, batch_size=500_000):
    cache = CACHE_DIR / f"{split}_inputs.npz"
    if cache.exists():
        data = dict(np.load(cache))
        print(f"{split}: loaded {len(data['label']):,} rows from cache")
        return data
    t0 = time.time()
    imps = pd.read_parquet(PROCESSED_DIR / f"{split}_impressions.parquet", columns=["impression_id", "history"])
    imps = imps.drop_duplicates("impression_id").reset_index(drop=True)
    imp_index = pd.Index(imps["impression_id"].astype(str))
    hc, hs = history_topics(imps["history"].tolist())

    parts = {k: [] for k in ("cat", "sub", "num", "label", "cold", "imp", "news")}
    for batch in pq.ParquetFile(PROCESSED_DIR / f"features_{split}.parquet").iter_batches(
            batch_size=batch_size, columns=INPUT_COLUMNS):
        df = batch.to_pandas()
        imp = imp_index.get_indexer(df["impression_id"].astype(str))
        assert (imp >= 0).all(), f"{split}: feature rows without a matching impression"
        assert df["label"].isin([0, 1]).all()
        parts["cat"].append(df["category_idx"].to_numpy(np.int16))
        parts["sub"].append(df["subcategory_idx"].to_numpy(np.int16))
        parts["num"].append(((get_numeric_matrix(df) - fit_mean) / fit_std).astype(np.float32))
        parts["label"].append(df["label"].to_numpy(np.int8))
        parts["cold"].append(df["cold_article"].to_numpy(np.int8))
        parts["imp"].append(imp.astype(np.int32))
        parts["news"].append(NEWS_INDEX.get_indexer(df["news_id"].astype(str)).astype(np.int32))
    data = {k: np.concatenate(v) for k, v in parts.items()}
    hist_n = imps["history"].fillna("").astype(str).str.split().str.len().to_numpy(np.int32)
    data.update(hc=hc, hs=hs, hist_n=hist_n, imp_ids=imps["impression_id"].to_numpy())
    if data["imp_ids"].dtype == object:
        data["imp_ids"] = data["imp_ids"].astype(str)

    expected = pq.ParquetFile(PROCESSED_DIR / CANDIDATE_FILES[split]).metadata.num_rows
    assert len(data["label"]) == expected, f"{split}: {len(data['label']):,} feature rows vs {expected:,} candidates"
    np.savez(cache, **data)
    print(f"{split}: {len(data['label']):,} rows | {len(np.unique(data['imp'])):,} impressions | "
          f"click rate {data['label'].mean():.4f} | {time.time() - t0:.0f}s")
    return data


fit = build_split_inputs("fit")
tune = build_split_inputs("tune")

fit: 12,894,210 rows | 1,801,231 impressions | click rate 0.2104 | 69s
tune: 17,400,106 rows | 431,517 impressions | click rate 0.0386 | 44s


### Training groups and the Tune subset
- **Training groups** are built once from all of Fit and reused by every run, so all configurations train on the same data.
- **Tune subset:** a fixed random set of Tune impressions (full slates), scored after every epoch to pick the best epoch. The full Tune day is used to compare configurations.

In [12]:
GROUPS, TARGETS, SKIPPED = five_way_groups_from_arrays(
    fit["imp"], fit["news"], fit["label"], np.random.default_rng(CFG["seed"]))
assert (fit["label"][GROUPS[np.arange(len(GROUPS)), TARGETS]] == 1).all()
assert (fit["label"][GROUPS].sum(axis=1) == 1).all()
assert (fit["imp"][GROUPS] == fit["imp"][GROUPS[:, :1]]).all()      # all 5 rows from one impression
print(f"Training groups: {len(GROUPS):,} (of {int(fit['label'].sum()):,} clicks) | skipped: {SKIPPED:,}")

def sample_impression_rows(data, n, seed):
    imps = np.unique(data["imp"])
    keep = np.random.default_rng(seed).choice(imps, size=min(n, len(imps)), replace=False)
    return np.flatnonzero(np.isin(data["imp"], keep))

TUNE_SUBSET = sample_impression_rows(tune, CFG["n_tune_imps"], CFG["seed"])
print(f"Tune subset for epoch selection: {len(np.unique(tune['imp'][TUNE_SUBSET])):,} impressions")

# Fit arrays on the GPU once, shared by all runs
FIT_GPU = {k: torch.as_tensor(fit[k], device=device) for k in ("cat", "sub", "num", "imp", "hc", "hs")}
GROUPS_GPU = torch.as_tensor(GROUPS, device=device)
TARGETS_GPU = torch.as_tensor(TARGETS, device=device)

Training groups: 2,445,153 (of 2,712,612 clicks) | skipped: 267,459
Tune subset for epoch selection: 20,000 impressions


## 8. Training loop with Tune epoch selection

- **One epoch:** all training groups in random order, batches of 128 groups, AdamW (weight decay 1e-4) and gradient clipping at 1.0.
- **After each epoch:** the Tune subset is scored. The epoch with the best Tune AUC is saved as `{run}_best.pt`; training stops after 2 epochs without improvement (at most 5 epochs).
- **Resume:** `{run}_last.pt` is saved after every epoch, so a stopped run continues where it left off.

In [13]:
def train_one_epoch(model, optimizer, batch_size, seed):
    model.train()
    gen = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(GROUPS_GPU), generator=gen).to(device)
    total_loss = 0.0
    for start in range(0, len(perm), batch_size):
        b = perm[start:start + batch_size]
        rows = GROUPS_GPU[b]                                   # [B, 5]
        imp = FIT_GPU["imp"][rows[:, 0]].long()
        # each candidate in a group shares the user's history
        user = model.user_topic(FIT_GPU["hc"][imp].long(), FIT_GPU["hs"][imp].long())
        user = user.repeat_interleave(rows.shape[1], dim=0)
        flat = rows.reshape(-1)
        scores = model.score(FIT_GPU["cat"][flat].long(), FIT_GPU["sub"][flat].long(), user,
                             FIT_GPU["num"][flat]).reshape(rows.shape)
        loss = five_way_loss(scores, TARGETS_GPU[b])
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(b)
    return total_loss / len(perm)


def train_model(run_name, cfg, numeric=NUMERICAL_COLUMNS, use_user_topic=True):
    seed_everything(cfg["seed"])
    model = FeatureMLP(n_categories, n_subcategories, cfg["hidden"], cfg["dropout"], numeric, use_user_topic).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    best_path, last_path = RUN_DIR / f"{run_name}_best.pt", RUN_DIR / f"{run_name}_last.pt"
    log, start_epoch, best_auc, bad_epochs = [], 1, -np.inf, 0

    if RESUME and last_path.exists():
        state = torch.load(last_path, map_location=device, weights_only=False)
        if state["architecture"] != model.architecture:
            raise ValueError(f"{last_path.name} does not match the current model.")
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        log, start_epoch = state["log"], state["epoch"] + 1
        best_auc, bad_epochs = state["best_auc"], state["bad_epochs"]
        print(f"{run_name}: resuming after epoch {state['epoch']}")

    for epoch in range(start_epoch, cfg["max_epochs"] + 1):
        if bad_epochs >= cfg["patience"]:
            break
        t0 = time.time()
        train_loss = train_one_epoch(model, optimizer, cfg["batch_size"], cfg["seed"] + epoch)
        tune_auc = mean_impression_auc(tune["imp"][TUNE_SUBSET], tune["label"][TUNE_SUBSET],
                                       score_rows(model, tune, TUNE_SUBSET))["mean_auc"]
        best_auc_new, saved = save_best_checkpoint(model, optimizer, epoch, tune_auc,
                                                   best_auc + 1e-4 if np.isfinite(best_auc) else -np.inf, best_path)
        if saved:
            best_auc, bad_epochs = tune_auc, 0
        else:
            bad_epochs += 1
        log.append({"epoch": epoch, "train_loss": train_loss, "tune_subset_auc": tune_auc,
                    "minutes": (time.time() - t0) / 60})
        print(f"{run_name} | epoch {epoch} | loss {train_loss:.4f} | tune-subset AUC {tune_auc:.4f}"
              f"{' | best' if saved else ''} | {log[-1]['minutes']:.1f} min", flush=True)
        state = {"epoch": epoch, "model_state_dict": model.state_dict(),
                 "optimizer_state_dict": optimizer.state_dict(), "log": log, "best_auc": best_auc,
                 "bad_epochs": bad_epochs, "architecture": model.architecture}
        torch.save(state, last_path.with_suffix(".partial.pt"))
        os.replace(last_path.with_suffix(".partial.pt"), last_path)

    best = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best["model_state_dict"])
    model.best_epoch = int(best["epoch"])
    pd.DataFrame(log).to_csv(RUN_DIR / f"{run_name}_log.csv", index=False)
    print(f"{run_name}: using epoch {model.best_epoch} (tune-subset AUC {best['tune_mean_auc']:.4f})")
    return model

## 9. Hyperparameter grid (Tune)

Each configuration is trained with the loop above and then scored on the **full** Tune day with `utils/metrics.py`. The configuration with the highest full Tune AUC becomes the final model. Dropout stays at 0.2.

In [14]:
def run_name(cfg):
    return f"mlp_lr{cfg['lr']:g}_h{'x'.join(map(str, cfg['hidden']))}"

def tune_frame(scores):
    return pd.DataFrame({"impression_id": tune["imp"], "label": tune["label"],
                         "is_cold": tune["cold"], "score": scores})

configs = [dict(CFG, lr=lr, hidden=h) for lr in GRID["lr"] for h in GRID["hidden"]] if RUN_GRID else [CFG]
grid_rows, models = [], {}
for cfg in configs:
    name = run_name(cfg)
    m = train_model(name, cfg)
    res = evaluate(tune_frame(score_rows(m, tune)), "score")
    grid_rows.append({"configuration": name, "lr": cfg["lr"], "hidden": str(cfg["hidden"]),
                      "best_epoch": m.best_epoch, **{f"tune_{k}": v for k, v in res.items()}})
    models[name] = (cfg, m)
grid = pd.DataFrame(grid_rows).sort_values("tune_auc", ascending=False)
grid.to_csv(RUN_DIR / "grid_tune.csv", index=False)
display(grid.round(4))

BEST = grid.iloc[0]["configuration"]
best_cfg, model = models[BEST]
models = {BEST: models[BEST]}
with open(RUN_DIR / "final_config.json", "w") as f:
    json.dump({**best_cfg, "run": BEST, "best_epoch": model.best_epoch, "trained_on": "fit 9-13 Nov"}, f, indent=2)
print("Final configuration:", BEST, "| epoch", model.best_epoch)

mlp_lr0.001_h64x32 | epoch 1 | loss 1.4175 | tune-subset AUC 0.6557 | best | 1.3 min
mlp_lr0.001_h64x32 | epoch 2 | loss 1.4012 | tune-subset AUC 0.6585 | best | 1.2 min
mlp_lr0.001_h64x32 | epoch 3 | loss 1.3966 | tune-subset AUC 0.6582 | 1.2 min
mlp_lr0.001_h64x32 | epoch 4 | loss 1.3936 | tune-subset AUC 0.6600 | best | 1.3 min
mlp_lr0.001_h64x32 | epoch 5 | loss 1.3920 | tune-subset AUC 0.6549 | 1.2 min
mlp_lr0.001_h64x32: using epoch 4 (tune-subset AUC 0.6600)
mlp_lr0.001_h128x64 | epoch 1 | loss 1.4077 | tune-subset AUC 0.6569 | best | 1.2 min
mlp_lr0.001_h128x64 | epoch 2 | loss 1.3884 | tune-subset AUC 0.6561 | 1.2 min
mlp_lr0.001_h128x64 | epoch 3 | loss 1.3829 | tune-subset AUC 0.6572 | best | 1.2 min
mlp_lr0.001_h128x64 | epoch 4 | loss 1.3800 | tune-subset AUC 0.6575 | best | 1.3 min
mlp_lr0.001_h128x64 | epoch 5 | loss 1.3779 | tune-subset AUC 0.6542 | 1.3 min
mlp_lr0.001_h128x64: using epoch 4 (tune-subset AUC 0.6575)
mlp_lr0.0003_h64x32 | epoch 1 | loss 1.4309 | tune-sub

,configuration,lr,hidden,best_epoch,tune_auc,tune_mrr,tune_ndcg@5,tune_ndcg@10,tune_n_impressions,tune_n_skipped
0,mlp_lr0.001_h64x32,0.0010,"(64, 32)",4,0.6620,0.3048,0.3293,0.3894,431517,0
1,mlp_lr0.001_h128x64,0.0010,"(128, 64)",4,0.6603,0.3051,0.3294,0.3901,431517,0
3,mlp_lr0.0003_h128x64,0.0003,"(128, 64)",1,0.6602,0.3033,0.3279,0.3886,431517,0
2,mlp_lr0.0003_h64x32,0.0003,"(64, 32)",1,0.6594,0.3031,0.3283,0.3885,431517,0


Final configuration: mlp_lr0.001_h64x32 | epoch 4


## 10. Results: training and testing performance

All splits are scored on **full slates** (every displayed article) with the shared `utils/metrics.py`.
- **Training:** 20,000 Fit impressions (9–13 Nov). `features_fit` only holds the sampled candidates, so the features of every displayed article in these impressions are rebuilt here with the same definitions as `step3_features`. The rebuilt values are first checked against `features_fit` on the sampled candidates; if they do not match, training performance falls back to the sampled slates and says so.
- **Tune:** 14 Nov (used for model selection). **Test:** 15 Nov (MIND dev), scored once.

In [15]:
# ---------- rebuild full-slate features for a sample of Fit impressions ----------
def id_filter(path, ids):
    # match the impression_id type stored in the file (int or string)
    is_int = pa.types.is_integer(pq.ParquetFile(path).schema_arrow.field("impression_id").type)
    return [("impression_id", "in", [int(x) for x in ids] if is_int else [str(x) for x in ids])]

def daily_display_counts(path):
    # displays per (UTC day, article) over all Fit impressions, from the full slates
    counts = Counter()
    for b in pq.ParquetFile(path).iter_batches(batch_size=100_000, columns=["time", "impressions"]):
        d = b.to_pandas()
        days = pd.to_datetime(d["time"], utc=True).dt.date
        for day, s in zip(days, d["impressions"]):
            for item in s.split():
                counts[(day, item.rsplit("-", 1)[0])] += 1
    return counts

def prior_counts_by_day(counts):
    days = sorted({d for d, _ in counts})
    running, prior = Counter(), {}
    for day in days:
        prior[day] = dict(running)              # displays on earlier days only
        for (d, nid), c in counts.items():
            if d == day:
                running[nid] += c
    return prior

def full_slate_features(imp_ids, prior):
    path = PROCESSED_DIR / "fit_impressions.parquet"
    imp = pq.read_table(path, columns=["impression_id", "time", "history", "impressions"],
                        filters=id_filter(path, imp_ids)).to_pandas().drop_duplicates("impression_id")
    emb = np.load(EMB_DIR / "news_embeddings.npy", mmap_mode="r")
    emb_row = {str(n): i for i, n in enumerate(pd.read_parquet(EMB_DIR / "news_embedding_ids.parquet")["news_id"])}
    first_seen = pd.read_parquet(PROCESSED_DIR / "step3_article_first_seen.parquet")
    first_seen = first_seen.assign(news_id=first_seen["news_id"].astype(str)).set_index("news_id")["first_seen_time"]
    news_cat = dict(zip(news_metadata["news_id"], news_metadata["category"]))
    rows = []
    for iid, t, h, s in zip(imp["impression_id"], imp["time"], imp["history"], imp["impressions"]):
        hist = h.split() if isinstance(h, str) and h.strip() else []
        cats = Counter(news_cat.get(x) for x in hist if pd.notna(news_cat.get(x)))
        prof = [emb_row[x] for x in hist if x in emb_row]
        prof = np.asarray(emb[prof], dtype=np.float32).mean(0) if prof else None
        if prof is not None:
            nrm = np.linalg.norm(prof)
            prof = prof / nrm if nrm > 0 else None
        p = prior[pd.to_datetime(t, utc=True).date()]
        for item in s.split():
            nid, lab = item.rsplit("-", 1)
            c = news_cat.get(nid)
            sim = float(np.clip(np.dot(prof, emb[emb_row[nid]]), -1, 1)) if prof is not None and nid in emb_row else 0.0
            rows.append((iid, nid, int(lab), t, len(hist),
                         cats.get(c, 0) / len(hist) if hist and pd.notna(c) else 0.0, sim, p.get(nid, 0)))
    df = pd.DataFrame(rows, columns=["impression_id", "news_id", "label", "time", "hist_len", "cat_match",
                                     "user_article_sim", "prior_count"])
    # exposure percentile within the full slate: average rank / (n - 1), 0.5 for one-article slates
    n = df.groupby("impression_id")["prior_count"].transform("size")
    r = df.groupby("impression_id")["prior_count"].rank(method="average") - 1
    df["slate_exposure_pct"] = np.where(n > 1, r / (n - 1).clip(lower=1), 0.5)
    df["category_idx"] = df["news_id"].map(news_category_lookup).fillna(0).astype(int)
    df["subcategory_idx"] = df["news_id"].map(news_subcategory_lookup).fillna(0).astype(int)
    age = pd.to_datetime(df["time"], utc=True) - pd.to_datetime(df["news_id"].map(first_seen), utc=True)
    df["log_article_age_hours"] = np.log1p(age.dt.total_seconds() / 3600)
    return df

used = fit["imp_ids"][np.unique(fit["imp"][GROUPS[:, 0]])]          # impressions used in training
train_ids = np.random.default_rng(CFG["seed"]).choice(used, size=min(CFG["n_train_slate_imps"], len(used)), replace=False)
prior = prior_counts_by_day(daily_display_counts(PROCESSED_DIR / "fit_impressions.parquet"))
full = full_slate_features(train_ids, prior)

# check the rebuilt features against step3_features on the sampled candidates of these impressions
fpath = PROCESSED_DIR / "features_fit.parquet"
stored = pq.read_table(fpath, columns=INPUT_COLUMNS, filters=id_filter(fpath, train_ids)).to_pandas()
for d in (full, stored):
    d["key_imp"], d["key_news"] = d["impression_id"].astype(str), d["news_id"].astype(str)
both = full.merge(stored, on=["key_imp", "key_news"], suffixes=("", "_stored"))
CHECK = ["category_idx", "subcategory_idx", "hist_len", "cat_match", "user_article_sim",
         "slate_exposure_pct", "log_article_age_hours"]
match = {c: float(np.isclose(both[c], both[f"{c}_stored"], atol=1e-3).mean()) for c in CHECK}
print(f"Rebuilt vs stored on {len(both):,} sampled candidates (share matching):", match)
FULL_SLATE_TRAIN = len(both) == len(stored) and min(match.values()) > 0.999
print("Training performance on full slates" if FULL_SLATE_TRAIN else
      "⚠️ Rebuilt features do not match step3_features: training performance uses the sampled slates")

if FULL_SLATE_TRAIN:
    train_data = {"num": ((get_numeric_matrix(full) - fit_mean) / fit_std).astype(np.float32),
                  "cat": full["category_idx"].to_numpy(), "sub": full["subcategory_idx"].to_numpy(),
                  "imp": pd.Index(fit["imp_ids"].astype(str)).get_indexer(full["key_imp"]),
                  "label": full["label"].to_numpy(np.int8), "hc": fit["hc"], "hs": fit["hs"]}
    train_rows = None
    train_keys = full[["impression_id", "news_id"]]
else:
    train_data = fit
    train_rows = np.flatnonzero(np.isin(fit["imp"], pd.Index(fit["imp_ids"].astype(str)).get_indexer(train_ids.astype(str))))
    train_keys = None

Rebuilt vs stored on 158,128 sampled candidates (share matching): {'category_idx': 1.0, 'subcategory_idx': 1.0, 'hist_len': 1.0, 'cat_match': 1.0, 'user_article_sim': 1.0, 'slate_exposure_pct': 1.0, 'log_article_age_hours': 1.0}
Training performance on full slates


In [16]:
def frame(data, scores, rows=None):
    rows = np.arange(len(data["label"])) if rows is None else rows
    out = pd.DataFrame({"impression_id": data["imp"][rows], "label": data["label"][rows], "score": scores})
    if "cold" in data:
        out["is_cold"] = data["cold"][rows]
    return out

def write_scores(split, scores, name):
    """Score file in the row order of features_{split}.parquet: impression_id, news_id, score."""
    path = SCORE_DIR / f"{name}.parquet"
    tmp = path.with_suffix(".partial.parquet")
    writer, start = None, 0
    for batch in pq.ParquetFile(PROCESSED_DIR / f"features_{split}.parquet").iter_batches(
            batch_size=500_000, columns=["impression_id", "news_id"]):
        table = pa.Table.from_batches([batch]).append_column("score", pa.array(scores[start:start + len(batch)]))
        start += len(batch)
        writer = writer or pq.ParquetWriter(tmp, table.schema, compression="zstd")
        writer.write_table(table)
    writer.close()
    assert start == len(scores), f"{split}: wrote {start:,} rows for {len(scores):,} scores"
    os.replace(tmp, path)
    print(f"saved {name}: {start:,} rows")

results = {}
s = score_rows(model, train_data, train_rows)
if FULL_SLATE_TRAIN:
    train_keys.assign(score=s).to_parquet(SCORE_DIR / f"{PREFIX}_train_sample.parquet", index=False)
    print(f"saved {PREFIX}_train_sample: {len(s):,} rows")
results["Training (20k Fit impressions)"] = evaluate(frame(train_data, s, train_rows), "score")

tune_scores = score_rows(model, tune)
write_scores("tune", tune_scores, f"{PREFIX}_tune")
results["Tune (14 Nov)"] = evaluate(tune_frame(tune_scores), "score")
tune_per_imp_final = evaluate(tune_frame(tune_scores), "score", per_impression=True)

test = build_split_inputs("eval")
test_scores = score_rows(model, test)
write_scores("eval", test_scores, f"{PREFIX}_eval")
test_df = frame(test, test_scores)
results["Test (15 Nov)"] = evaluate(test_df, "score")

overall = pd.DataFrame(results).T[["auc", "mrr", "ndcg@5", "ndcg@10", "n_impressions", "n_skipped"]]
overall[["n_impressions", "n_skipped"]] = overall[["n_impressions", "n_skipped"]].astype(int)
overall.to_csv(RUN_DIR / "overall.csv")
print(f"Final model: {BEST}, epoch {model.best_epoch}")
display(overall.round(4))

saved step3_feature_mlp_train_sample: 846,159 rows
saved step3_feature_mlp_tune: 17,400,106 rows
eval: 14,085,557 rows | 376,471 impressions | click rate 0.0408 | 34s
saved step3_feature_mlp_eval: 14,085,557 rows
Final model: mlp_lr0.001_h64x32, epoch 4


,auc,mrr,ndcg@5,ndcg@10,n_impressions,n_skipped
Training (20k Fit impressions),0.7052,0.3011,0.3308,0.3953,20000,0
Tune (14 Nov),0.6620,0.3048,0.3293,0.3894,431517,0
Test (15 Nov),0.6443,0.3068,0.3362,0.3989,376471,0


### Test results by segment
- **Cold vs warm:** impressions grouped by their clicked articles (cold = every clicked article had fewer than 10 displays on earlier days, warm = none did, mixed = some of each). Compare segments by AUC, since MRR and nDCG also depend on slate length.
- **By history length:** the user-side cold start.

In [17]:
seg = evaluate_by_segment(test_df, "score", "is_cold")[["n_impressions", "auc", "mrr", "ndcg@10"]]
seg = seg.reindex(["all", "cold", "warm", "mixed"]).dropna(how="all")
seg.to_csv(RUN_DIR / "eval_by_segment.csv")
display(seg.round(4))

per = evaluate(test_df, "score", per_impression=True)
per["history"] = pd.cut(test["hist_n"][per["impression_id"].to_numpy()], [-1, 0, 5, 20, 50, np.inf],
                        labels=["0 (no history)", "1-5", "6-20", "21-50", "over 50"])
by_hist = per.groupby("history", observed=False).agg(impressions=("auc", "size"), auc=("auc", "mean"), mrr=("mrr", "mean"))
by_hist.to_csv(RUN_DIR / "eval_by_history.csv")
display(by_hist.round(4))

,n_impressions,auc,mrr,ndcg@10
all,376471,0.6443,0.3068,0.3989
cold,77199,0.6763,0.4187,0.5147
warm,262807,0.6301,0.2901,0.3748
mixed,36465,0.6788,0.1903,0.3274


,impressions,auc,mrr
history,,,
0 (no history),11270,0.5405,0.2381
1-5,55262,0.5994,0.3204
6-20,131368,0.6402,0.3172
21-50,103557,0.6645,0.3057
over 50,75014,0.6721,0.2907


## 11. Ablations (Tune)

Each ablation retrains the final configuration with one input group removed and is compared with the final model on the **full Tune day**, impression by impression:
- **without user × news features** (content similarity and category match): how much personalisation comes from the match features;
- **without the user topic embedding:** whether the learned taste vector adds anything beyond the match features.

The standard error (SE) is the standard deviation of the per-impression differences ÷ √(number of impressions); a difference larger than about 2 SE is treated as significant. Test is not used.

In [18]:
ABLATIONS = {
    "without user x news features": dict(numeric=[c for c in NUMERICAL_COLUMNS if c not in USER_NEWS_COLUMNS]),
    "without user topic embedding": dict(use_user_topic=False),
}

def paired(a, b):
    d = (a - b).dropna()
    return d.mean(), d.std(ddof=1) / np.sqrt(len(d))

if RUN_ABLATIONS:
    seg_of = label_impression_segment(tune_frame(tune_scores), "is_cold")
    base = tune_per_imp_final.set_index("impression_id")
    cold = seg_of.reindex(base.index) == "cold"
    rows = {"final model": {"auc": base.auc.mean(), "mrr": base.mrr.mean(), "ndcg@10": base["ndcg@10"].mean(),
                            "cold_click_auc": base.auc[cold].mean()}}
    for name, change in ABLATIONS.items():
        m = train_model(f"{BEST}_abl_" + name.replace(" ", "_"), best_cfg, **change)
        per = evaluate(tune_frame(score_rows(m, tune)), "score", per_impression=True).set_index("impression_id")
        is_cold = cold.reindex(per.index, fill_value=False)
        d, se = paired(per.auc, base.auc)
        dc, sec = paired(per.auc[is_cold], base.auc[cold])
        rows[name] = {"auc": per.auc.mean(), "mrr": per.mrr.mean(), "ndcg@10": per["ndcg@10"].mean(),
                      "cold_click_auc": per.auc[is_cold].mean(),
                      "AUC_vs_final": d, "SE": se, "cold_AUC_vs_final": dc, "cold_SE": sec}
        del m
    abl = pd.DataFrame(rows).T
    abl.to_csv(RUN_DIR / "ablation_tune.csv")
    display(abl.round(4))

mlp_lr0.001_h64x32_abl_without_user_x_news_features | epoch 1 | loss 1.4710 | tune-subset AUC 0.6317 | best | 1.2 min
mlp_lr0.001_h64x32_abl_without_user_x_news_features | epoch 2 | loss 1.4486 | tune-subset AUC 0.6314 | 1.2 min
mlp_lr0.001_h64x32_abl_without_user_x_news_features | epoch 3 | loss 1.4427 | tune-subset AUC 0.6288 | 1.3 min
mlp_lr0.001_h64x32_abl_without_user_x_news_features: using epoch 1 (tune-subset AUC 0.6317)
mlp_lr0.001_h64x32_abl_without_user_topic_embedding | epoch 1 | loss 1.4257 | tune-subset AUC 0.6515 | best | 1.1 min
mlp_lr0.001_h64x32_abl_without_user_topic_embedding | epoch 2 | loss 1.4115 | tune-subset AUC 0.6504 | 1.0 min
mlp_lr0.001_h64x32_abl_without_user_topic_embedding | epoch 3 | loss 1.4080 | tune-subset AUC 0.6505 | 1.1 min
mlp_lr0.001_h64x32_abl_without_user_topic_embedding: using epoch 1 (tune-subset AUC 0.6515)


,auc,mrr,ndcg@10,cold_click_auc,AUC_vs_final,SE,cold_AUC_vs_final,cold_SE
final model,0.6620,0.3048,0.3894,0.6565,NaN,NaN,NaN,NaN
without user x news features,0.6324,0.2815,0.3618,0.6135,-0.0296,0.0004,-0.0430,0.0009
without user topic embedding,0.6544,0.3008,0.3854,0.6407,-0.0076,0.0002,-0.0158,0.0006


## 12. Hand-off

| Output | Location | Used by |
|---|---|---|
| Tune and Test scores | `data/scores/step3_feature_mlp_tune.parquet`, `step3_feature_mlp_eval.parquet` | Step 6 (model comparison) |
| Fit-impression scores | `data/scores/step3_feature_mlp_train_sample.parquet` | Training vs test comparison |
| Grid, results, ablations, logs | `outputs/step3_feature_mlp/` | Report |